# RecursiveMAS — Latent channel smoke test (Colab)

Week-1 **go/no-go** probe from [`PROPOSAL_LONGFORM.md`](../PROPOSAL_LONGFORM.md): do released **RecursiveLink** adapters carry signal on long inputs, or collapse toward identity (`cos(in,out) → 1`)?

## Before you run
1. **Runtime → Change runtime type → GPU** (T4 is enough for `t4_minimal`; use **A100** for `a100_extended`).
2. Colab Pro: optional **High RAM** if `a100_extended` OOMs.
3. First run downloads ~6–12 GB (code model + outer links). Cache on Drive (cell below) to avoid re-downloading.

## Run order (important)
Run cells **top to bottom** on first setup: **Config → Drive → Install** (restarts runtime). After reconnect, run **GPU check → Helpers → …** — GPU check restores defaults; **Config is optional** unless you changed paths.

The **install** cell restarts the runtime automatically (fixes Colab `numpy._center` errors). After it restarts, **run all cells again** from **GPU check** downward (Config + Drive + Install can be skipped if already done).

## Profiles
| Profile | GPU | What it runs |
|---|---|---|
| `t4_minimal` | T4 / L4 | Code expert + `outer_2s` only; short vs long prompt |
| `a100_extended` | A100 40GB+ | Above + `sequential_light` 3-agent one recursion round |

**Cursor cannot execute this notebook** — run cells here, then paste the printed **VERDICT** block back into Cursor.

Repo: clone from GitHub below, or upload your local `RecursiveMAS` folder and set `REPO_DIR` manually.

In [ ]:
# @title Configuration
import os
from pathlib import Path

# auto picks t4_minimal on <=16GB VRAM else a100_extended
RUN_PROFILE = "auto"  # auto | t4_minimal | a100_extended

REPO_URL = "https://github.com/RecursiveMAS/RecursiveMAS.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/RecursiveMAS")

LATENT_STEPS = 32
BATCH_SIZE = 1
DTYPE = "auto"  # float16/bfloat16 on GPU
OUTER_DTYPE = "auto"
TRUST_REMOTE_CODE = True

# Persist Hugging Face cache on Google Drive (recommended)
USE_DRIVE_CACHE = True
DRIVE_HF_HOME = "/content/drive/MyDrive/huggingface"

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MAS_FORCE_DISABLE_TORCHVISION"] = "1"

In [ ]:
# @title Mount Drive + HF cache (optional)
import os
from pathlib import Path

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    Path(DRIVE_HF_HOME).mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = DRIVE_HF_HOME
    print("HF_HOME =", os.environ["HF_HOME"])
else:
    print("Using ephemeral Colab cache under ~/.cache/huggingface")

In [ ]:
# @title Clone repo + install dependencies (restarts runtime when done)
import os
import subprocess
import sys
from pathlib import Path

if not REPO_DIR.is_dir():
    subprocess.check_call([
        "git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR),
    ])
else:
    print("REPO_DIR already exists:", REPO_DIR)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

pip = [sys.executable, "-m", "pip", "install", "-q"]
req = REPO_DIR / "requirements.txt"
req_lines = [
    ln.strip()
    for ln in req.read_text(encoding="utf-8").splitlines()
    if ln.strip() and not ln.strip().startswith("#") and not ln.lower().startswith("numpy")
]

# Colab preloads NumPy in the kernel; pip-only upgrades leave a broken mix
# (ImportError: cannot import name '_center' from numpy._core.umath).
subprocess.check_call(pip + ["--upgrade", "pip"])
subprocess.check_call(pip + ["--force-reinstall", "numpy==2.2.6"])
for spec in req_lines:
    subprocess.check_call(pip + [spec])

subprocess.check_call([
    sys.executable, "-c",
    "import numpy as np; import torch; print('numpy', np.__version__, 'torch', torch.__version__)",
])

print("\nRestarting Colab runtime so NumPy/Torch load cleanly (expected).")
print("When it reconnects: Runtime → Run all, or run from 'GPU check' downward.")
os._exit(0)

In [ ]:
# @title GPU check + profile selection (run AFTER install + runtime restart)
def bootstrap_notebook_config():
    """Restore defaults after Colab runtime restart (install cell clears kernel)."""
    import os
    from pathlib import Path
    defaults = {
        "REPO_DIR": Path("/content/RecursiveMAS"),
        "RUN_PROFILE": "auto",
        "LATENT_STEPS": 32,
        "BATCH_SIZE": 1,
        "DTYPE": "auto",
        "OUTER_DTYPE": "auto",
        "TRUST_REMOTE_CODE": True,
        "REPO_URL": "https://github.com/RecursiveMAS/RecursiveMAS.git",
        "REPO_BRANCH": "main",
        "USE_DRIVE_CACHE": True,
        "DRIVE_HF_HOME": "/content/drive/MyDrive/huggingface",
    }
    g = globals()
    for key, val in defaults.items():
        if key not in g:
            g[key] = val
    if not isinstance(g["REPO_DIR"], Path):
        g["REPO_DIR"] = Path(g["REPO_DIR"])
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    os.environ.setdefault("MAS_FORCE_DISABLE_TORCHVISION", "1")
    return g

bootstrap_notebook_config()

import sys
from pathlib import Path

bootstrap_notebook_config()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import os
os.chdir(REPO_DIR)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Use Runtime → Change runtime type → GPU.")

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}  VRAM: {vram_gb:.1f} GB")

profile = RUN_PROFILE
if profile == "auto":
    profile = "a100_extended" if vram_gb >= 35 else "t4_minimal"
    print(f"RUN_PROFILE=auto → using {profile}")
elif profile == "a100_extended" and vram_gb < 24:
    print("WARNING: a100_extended may OOM on this GPU; fall back to t4_minimal if needed.")
else:
    print(f"Using RUN_PROFILE={profile}")

SELECTED_PROFILE = profile


### After the install cell
Colab disconnects and reconnects. **Run the GPU check cell** (it restores `REPO_DIR` and other defaults — no need to re-run Config unless you changed paths) (or use *Runtime → Run all*). If you still see a NumPy error, use *Runtime → Restart session*, then run Config → Drive → GPU check → Helpers → … (skip Install if packages are already there).

In [ ]:
# @title Helpers (cosine stats + outer-link-only probe)
def bootstrap_notebook_config():
    """Restore defaults after Colab runtime restart (install cell clears kernel)."""
    import os
    from pathlib import Path
    defaults = {
        "REPO_DIR": Path("/content/RecursiveMAS"),
        "RUN_PROFILE": "auto",
        "LATENT_STEPS": 32,
        "BATCH_SIZE": 1,
        "DTYPE": "auto",
        "OUTER_DTYPE": "auto",
        "TRUST_REMOTE_CODE": True,
        "REPO_URL": "https://github.com/RecursiveMAS/RecursiveMAS.git",
        "REPO_BRANCH": "main",
        "USE_DRIVE_CACHE": True,
        "DRIVE_HF_HOME": "/content/drive/MyDrive/huggingface",
    }
    g = globals()
    for key, val in defaults.items():
        if key not in g:
            g[key] = val
    if not isinstance(g["REPO_DIR"], Path):
        g["REPO_DIR"] = Path(g["REPO_DIR"])
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    os.environ.setdefault("MAS_FORCE_DISABLE_TORCHVISION", "1")
    return g

bootstrap_notebook_config()


from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

bootstrap_notebook_config()

# Ensure repo root is on path (needed after runtime restart)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

import numpy as np
print("numpy", np.__version__)

import torch
import torch.nn.functional as F

from hf_resolver import resolve_inner_adapter, resolve_outer_paths, snapshot_repo
from inference_utils import inference_mas as mas
from inference_utils.inference_mas_mixture import run_hie_expert_latent_stage
from modeling import CrossModelAdapter

CODE_REPO = "RecursiveMAS/Mixture-Code-Qwen2.5-Coder-3B"
OUTER_REPO = "RecursiveMAS/Mixture-Outerlinks"
SEQUENTIAL_LIGHT = {
    "planner": "RecursiveMAS/Sequential-Light-Planner-Qwen3-1.7B",
    "critic": "RecursiveMAS/Sequential-Light-Critic-Llama3.2-1B",
    "solver": "RecursiveMAS/Sequential-Light-Solver-Qwen2.5-Math-1.5B",
    "outer": "RecursiveMAS/Sequential-Light-Outerlinks",
}


def resolve_dtype(name: str, device: torch.device) -> torch.dtype:
    resolved = mas.resolve_dtype(name)
    # mas.resolve_dtype("auto") returns the string "auto", not a torch.dtype
    if isinstance(resolved, torch.dtype):
        return resolved
    if device.type == "cuda":
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32


def latent_cosine_stats(input_tensor: torch.Tensor, output_tensor: torch.Tensor) -> Dict[str, float]:
    """Per-token cosine between outer-link input and output; pool with mean."""
    if input_tensor.dim() == 2:
        input_tensor = input_tensor.unsqueeze(0)
        output_tensor = output_tensor.unsqueeze(0)
    cos = F.cosine_similarity(input_tensor.float(), output_tensor.float(), dim=-1)
    flat = cos.reshape(-1)
    return {
        "cos_mean": float(flat.mean().item()),
        "cos_std": float(flat.std(unbiased=False).item()) if flat.numel() > 1 else 0.0,
        "cos_min": float(flat.min().item()),
        "cos_max": float(flat.max().item()),
        "num_tokens": int(flat.numel()),
    }


@torch.no_grad()
def probe_outer_weights_only(
    outer_path: Path,
    in_dim: int,
    out_dim: int,
    device: torch.device,
    outer_dtype: torch.dtype,
    seq_lens: Tuple[int, ...] = (8, 32, 128),
) -> List[Dict[str, Any]]:
    """Tier-0: adapter geometry without loading any LLM."""
    adapter = mas.load_outer_adapter_module(
        adapter_path=str(outer_path),
        in_dim=in_dim,
        out_dim=out_dim,
        adapter_type="outer_ln_res_adapter",
        device=device,
        dtype=outer_dtype,
    )
    rows = []
    for seq_len in seq_lens:
        x = torch.randn(1, seq_len, in_dim, device=device, dtype=outer_dtype)
        y = mas.run_outer_adapter(adapter, x, output_dtype=outer_dtype)
        stats = latent_cosine_stats(x, y)
        stats["probe"] = "synthetic"
        stats["seq_len"] = seq_len
        stats["outer_path"] = str(outer_path)
        rows.append(stats)
    mas.release_resources(adapter)
    return rows


@torch.no_grad()
def probe_code_expert_latent(
    code_dir: Path,
    outer_2s_path: Path,
    question: str,
    label: str,
    device: torch.device,
    model_dtype: torch.dtype,
    outer_dtype: torch.dtype,
    latent_steps: int,
    batch_size: int,
) -> Dict[str, Any]:
    """Tier-1: full code-expert latent rollout + outer_2s (code → summarizer)."""
    latents = run_hie_expert_latent_stage(
        stage_name="hie_code_expert",
        model_name_or_path=str(code_dir),
        questions=[question],
        role="hie_code_expert",
        inner_aligner_path=str(resolve_inner_adapter(code_dir, None)),
        outer_path=str(outer_2s_path),
        outer_type="outer_ln_res_adapter",
        latent_steps=latent_steps,
        batch_size=batch_size,
        device=device,
        model_dtype=model_dtype,
        outer_dtype=outer_dtype,
        trust_remote_code=TRUST_REMOTE_CODE,
        inner_adapter_type_fallback="ln_res_adapter",
        enable_thinking=False,
        mas_task="code",
        task_types=["complete"],
        fn_names=[None],
        feedback_latents=None,
    )
    latent = latents[0]
    return {
        "label": label,
        "prompt_chars": len(question),
        "latent_shape": list(latent.shape),
        "latent_norm_mean": float(latent.float().norm(dim=-1).mean().item()),
        "latent_std_mean": float(latent.float().std(dim=-1).mean().item()),
    }


@torch.no_grad()
def probe_code_expert_with_outer_cosine(
    code_dir: Path,
    outer_2s_path: Path,
    question: str,
    label: str,
    device: torch.device,
    model_dtype: torch.dtype,
    outer_dtype: torch.dtype,
    latent_steps: int,
) -> Dict[str, Any]:
    """Re-run forward hooks: measure cos(inner_latent, outer_mapped) on real prompts."""
    model, tokenizer = mas.load_agent_model_and_tokenizer(
        model_name_or_path=str(code_dir),
        device=device,
        dtype=model_dtype,
        trust_remote_code=TRUST_REMOTE_CODE,
        agent_name="hie_code_expert",
    )
    embed_layer = model.get_input_embeddings()
    embed_dtype = embed_layer.weight.dtype
    hidden_size = embed_layer.weight.size(-1)

    inner = mas.load_inner_adapter_module(
        adapter_path=str(resolve_inner_adapter(code_dir, None)),
        hidden_size=hidden_size,
        device=device,
        dtype=model_dtype,
        fallback_adapter_type="ln_res_adapter",
    )
    out_dim = mas.infer_outer_adapter_out_dim_from_file(str(outer_2s_path))
    outer = mas.load_outer_adapter_module(
        adapter_path=str(outer_2s_path),
        in_dim=hidden_size,
        out_dim=out_dim,
        adapter_type="outer_ln_res_adapter",
        device=device,
        dtype=outer_dtype,
    )

    from prompts import build_hie_expert_prompt

    user_prompt = build_hie_expert_prompt(
        question, "hie_code_expert", mas_task="code", task_type="complete"
    )
    prompt_ids = mas.render_chat_prompt_ids(tokenizer, user_prompt, enable_thinking=False)
    input_ids, attention_mask = mas.pad_left_ids([prompt_ids], pad_id=tokenizer.pad_token_id, device=device)
    input_embeds = embed_layer(input_ids)

    hidden_rollout = mas.autoregressive_latent_rollout(
        model=model,
        rollout_inner_adapter=inner,
        input_embeds=input_embeds,
        attention_mask=attention_mask,
        latent_steps=latent_steps,
    )
    self_latent = mas.run_inner_adapter(inner, hidden_rollout, output_dtype=embed_dtype)
    mapped = mas.run_outer_adapter(outer, self_latent, output_dtype=torch.float32)
    stats = latent_cosine_stats(self_latent, mapped)
    stats.update({
        "label": label,
        "prompt_chars": len(question),
        "prompt_tokens": int(attention_mask.sum().item()),
        "latent_steps": latent_steps,
        "outer_path": str(outer_2s_path),
    })
    mas.release_resources(model, tokenizer, inner, outer)
    return stats


def interpret_verdict(rows: List[Dict[str, Any]]) -> str:
    real = [r for r in rows if r.get("probe") != "synthetic"]
    if not real:
        real = rows
    means = [r["cos_mean"] for r in real if "cos_mean" in r]
    if not means:
        return "INCONCLUSIVE: no cosine rows"
    avg = sum(means) / len(means)
    if avg > 0.98:
        return f"FAIL (collapse risk): mean cos={avg:.4f} — outer link nearly identity; latent channel may carry little new information."
    if avg > 0.95:
        return f"WARN: mean cos={avg:.4f} — weak transformation; consider OOD fine-tune or smaller latent_steps."
    return f"PASS (probe): mean cos={avg:.4f} — outer link is actively transforming latents; proceed to long-form harness work."

print("Helpers loaded.")

In [ ]:
# @title Test prompts (short vs long code spec)
SHORT_CODE_PROMPT = """Write a complete program that reads from stdin and prints to stdout.
The programming problem is:
Given n lines of integers, print the sum of each line.
"""

# Simulates a longer multi-module spec (stress prompt tokens, not 20k-word generation yet)
LONG_CODE_PROMPT = SHORT_CODE_PROMPT + "\n" + """
Project context (read carefully; consistency matters across modules):
- package `app` with subpackages `api`, `core`, `storage`, `cli`.
- Module `api.routes` defines REST handlers; must stay compatible with `core.models`.
- Module `core.models` uses dataclasses for User, Session, Document; versioning field required.
- Module `storage.db` provides connection pooling and migrations; do not break public API.
- Module `cli.main` orchestrates ingest, index, and query commands.
Constraints:
1. All public functions require type hints and docstrings.
2. Logging via `core.logging` only; no print debugging in library code.
3. Error types inherit from `core.errors.AppError`.
4. When adding a new route, update OpenAPI schema and integration tests.
5. Symbol `parse_config` in `core.config` is used by both `api` and `cli`; keep signature stable.
""" * 8  # repeat block to inflate context

print("short chars", len(SHORT_CODE_PROMPT), "long chars", len(LONG_CODE_PROMPT))

In [ ]:
# @title Run smoke test
bootstrap_notebook_config()

import gc
import json
from datetime import datetime, timezone

device = torch.device("cuda")
model_dtype = resolve_dtype(DTYPE, device)
outer_dtype = resolve_dtype(OUTER_DTYPE, device)
if device.type == "cpu" and model_dtype in {torch.float16, torch.bfloat16}:
    model_dtype = torch.float32
    outer_dtype = torch.float32

print("Downloading Mixture code model + outer links (first run only)...")
code_dir = snapshot_repo(CODE_REPO)
outer_dir = snapshot_repo(OUTER_REPO)
outer_paths = resolve_outer_paths(outer_dir, task=None)
outer_2s = outer_paths["outer_2s"]
print("code_dir:", code_dir)
print("outer_2s:", outer_2s)

# Infer dims from code model config without full generate pass
probe_model, probe_tok = mas.load_agent_model_and_tokenizer(
    model_name_or_path=str(code_dir),
    device=device,
    dtype=model_dtype,
    trust_remote_code=TRUST_REMOTE_CODE,
    agent_name="probe",
)
hidden_size = int(probe_model.get_input_embeddings().weight.size(-1))
out_dim = mas.infer_outer_adapter_out_dim_from_file(str(outer_2s))
mas.release_resources(probe_model, probe_tok)
gc.collect()
torch.cuda.empty_cache()

results: List[Dict[str, Any]] = []

print("\n=== Tier 0: synthetic outer-link cosine (no LLM) ===")
results.extend(
    probe_outer_weights_only(
        outer_2s, hidden_size, out_dim, device, outer_dtype, seq_lens=(8, 32, 128)
    )
)
for r in results:
    if r.get("probe") == "synthetic":
        print(r)

print("\n=== Tier 1: code expert — short prompt ===")
results.append(
    probe_code_expert_with_outer_cosine(
        code_dir, outer_2s, SHORT_CODE_PROMPT, "short",
        device, model_dtype, outer_dtype, LATENT_STEPS,
    )
)
print(results[-1])
gc.collect()
torch.cuda.empty_cache()

print("\n=== Tier 1: code expert — long prompt ===")
results.append(
    probe_code_expert_with_outer_cosine(
        code_dir, outer_2s, LONG_CODE_PROMPT, "long",
        device, model_dtype, outer_dtype, LATENT_STEPS,
    )
)
print(results[-1])

if SELECTED_PROFILE == "a100_extended":
    print("\n=== Tier 2: sequential_light one recursion round (planner → critic outer_12) ===")
    gc.collect()
    torch.cuda.empty_cache()
    sl_outer_dir = snapshot_repo(SEQUENTIAL_LIGHT["outer"])
    sl_outer = resolve_outer_paths(sl_outer_dir, task="math")
    planner_dir = snapshot_repo(SEQUENTIAL_LIGHT["planner"])
    planner_inner = resolve_inner_adapter(planner_dir, "math")
    try:
        lat12 = mas.run_planner_latent_stage(
            model_name_or_path=str(planner_dir),
            questions=["What is 17 * 23? Show reasoning."],
            agent1_inner_aligner_path=str(planner_inner),
            outer_12_path=str(sl_outer["outer_12"]),
            outer_12_type="outer_ln_res_adapter",
            latent_steps=min(LATENT_STEPS, 16),
            batch_size=1,
            device=device,
            model_dtype=model_dtype,
            outer_dtype=outer_dtype,
            trust_remote_code=TRUST_REMOTE_CODE,
            inner_adapter_type_fallback="ln_res_adapter",
            enable_thinking=False,
        )
        results.append({
            "label": "sequential_light_planner_outer12",
            "latent_shape": list(lat12[0].shape),
            "latent_norm_mean": float(lat12[0].float().norm(dim=-1).mean().item()),
        })
        print(results[-1])
    except Exception as exc:
        print("Tier 2 skipped:", exc)
        results.append({"label": "sequential_light", "error": str(exc)})

verdict = interpret_verdict(results)
summary = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": gpu_name,
    "vram_gb": round(vram_gb, 2),
    "profile": SELECTED_PROFILE,
    "latent_steps": LATENT_STEPS,
    "verdict": verdict,
    "results": results,
}

out_path = REPO_DIR / "notebooks" / "smoke_results.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("\n" + "=" * 72)
print("VERDICT:", verdict)
print("Saved:", out_path)
print("=" * 72)
print("\nPaste the VERDICT line + short/long cos_mean values back into Cursor.")

## How to read results

| `cos_mean` (real prompts) | Meaning |
|---|---|
| **> 0.98** | Likely **collapse** — outer link ≈ identity; long-form latent MAS needs retrained links or a new training pipeline |
| **0.95 – 0.98** | Weak channel — proceed cautiously; OOD fine-tune likely required |
| **< 0.95** | Channel is **transforming** latents — reasonable to invest in long-form harness + AgentWrite text baseline |

Compare **short** vs **long** `cos_mean` and `prompt_tokens`. If long prompts push `cos_mean` toward 1.0, length is eroding the channel (supports RQ2 in `PROPOSAL_LONGFORM.md`).

Synthetic (Tier 0) cosines on random noise are **not** sufficient alone — they only show the adapter isn't literally zero on random input.